# Agentic Sidekick with LangGraph

This notebook implements an agentic workflow using LangGraph that goes beyond a simple chatbot by introducing structured reasoning, tool usage, and self-evaluation.

The system follows a multi-step process:
- **Planner**: Breaks down the user’s request into actionable steps
- **Worker**: Executes the task using an LLM and available tools
- **Tools**: Enables capabilities like code execution (and optionally search)
- **Evaluator**: Reviews the output against defined success criteria and decides whether to retry or finish

The workflow is built as a graph with conditional routing, allowing the agent to loop, improve its responses, and request clarification when needed.

This project demonstrates how to build a self-improving agent that can plan, act, and evaluate — forming the foundation for more advanced AI assistants.

In [ ]:
!pip install langgraph langchain langchain-openai langchain-community langchain-experimental python-dotenv

In [ ]:

import os
import gradio as gr
import uuid
from dotenv import load_dotenv
from typing import Annotated, List, Optional, Any, Dict
from typing_extensions import TypedDict
from datetime import datetime

# LangGraph / LangChain
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool

# Optional tools
from langchain_experimental.tools import PythonREPLTool
from langchain_community.utilities import GoogleSerperAPIWrapper

from pydantic import BaseModel, Field

load_dotenv()

MODEL = "gpt-4o-mini"  # or your model


In [ ]:
load_dotenv(override=True)

#### Tools

In [ ]:
serper = GoogleSerperAPIWrapper()

def search_tool_func(query: str):
    return serper.run(query)

search_tool = Tool(
    name="search",
    func=search_tool_func,
    description="Search the web for information"
)

python_tool = PythonREPLTool()

tools = [search_tool, python_tool]

#### State

In [ ]:
class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool
    plan: Optional[Any]
    final_worker_response: Optional[str]

#### Schemas

In [ ]:
class PlanOutput(BaseModel):
    steps: List[str]

class EvaluatorOutput(BaseModel):
    feedback: str
    success_criteria_met: bool
    user_input_needed: bool


In [ ]:
async def setup():
    sidekick = Sidekick()
    await sidekick.setup()
    return sidekick


async def get_clarifying_questions(message, success_criteria):
    """Generate exactly 3 clarifying questions from the user's request."""
    if not message or not str(message).strip():
        return " Enter a request first, then click again. "
    llm = ChatOpenAI(model=MODEL, base_url=openrouter_base_url, api_key=openrouter_api_key).with_structured_output(ClarifyingQuestions)
    prompt = (
        f"User request:\n{message}\n\nSuccess criteria:\n{success_criteria or '(none given)'}\n\n"
        "Generate exactly 3 short clarifying questions (scope, depth, constraints)."
    )
    out = llm.invoke([HumanMessage(content=prompt)])
    return "### Clarifying questions\n" + "\n".join(f"{i + 1}. {q}" for i, q in enumerate(out.questions))


async def process_message(sidekick, message, success_criteria, clarifications, history):
    if sidekick is None:
        return history + [{"role": "assistant", "content": "Sidekick failed to initialize. Please reset."}], None
    results = await sidekick.run_superstep(message, success_criteria, clarifications, history)
    return results, sidekick

In [ ]:
async def reset(sidekick):
    if sidekick is not None:
        await sidekick.aclose()
    new_sidekick = Sidekick()
    await new_sidekick.setup()
    return "", "", "", "", [], new_sidekick


def free_resources(sidekick):
    print("Cleaning up")
    try:
        if sidekick:
            sidekick.cleanup()
    except Exception as e:
        print(f"Exception during cleanup: {e}")

In [ ]:
worker_llm = ChatOpenAI(model=MODEL)
worker_llm_with_tools = worker_llm.bind_tools(tools)

planner_llm = ChatOpenAI(model=MODEL).with_structured_output(PlanOutput)

evaluator_llm = ChatOpenAI(model=MODEL).with_structured_output(EvaluatorOutput)

#### Planner Node

In [ ]:

def planner(state: State):
    user_input = state["messages"][-1].content

    prompt = f"""
    Break this task into steps:
    {user_input}
    """

    result = planner_llm.invoke([HumanMessage(content=prompt)])

    return {
        "messages": [AIMessage(content=f"Plan: {result.steps}")],
        "plan": result
    }

#### Worker Node

In [ ]:

def worker(state: State):
    plan = state.get("plan")

    system_prompt = f"""
    You are an assistant.
    Follow this plan:
    {plan.steps if plan else "No plan"}

    Success criteria:
    {state["success_criteria"]}
    """

    messages = [SystemMessage(content=system_prompt)] + state["messages"]

    response = worker_llm_with_tools.invoke(messages)

    return {
        "messages": [response],
        "final_worker_response": response.content
    }

In [ ]:

def worker_router(state: State):
    last = state["messages"][-1]

    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "evaluator"

#### Evaluator

In [ ]:

def evaluator(state: State):
    last_response = state["final_worker_response"]

    prompt = f"""
    Evaluate this response:

    Response:
    {last_response}

    Criteria:
    {state["success_criteria"]}
    """

    result = evaluator_llm.invoke([HumanMessage(content=prompt)])

    return {
        "messages": [AIMessage(content=f"Feedback: {result.feedback}")],
        "feedback_on_work": result.feedback,
        "success_criteria_met": result.success_criteria_met,
        "user_input_needed": result.user_input_needed
    }

In [ ]:
def route_after_evaluation(state: State):
    if state["success_criteria_met"] or state["user_input_needed"]:
        return "END"
    return "worker"

#### Graph

In [ ]:

memory = MemorySaver()

graph_builder = StateGraph(State)

graph_builder.add_node("planner", planner)
graph_builder.add_node("worker", worker)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_node("evaluator", evaluator)

graph_builder.add_edge(START, "planner")
graph_builder.add_edge("planner", "worker")

graph_builder.add_conditional_edges(
    "worker",
    worker_router,
    {"tools": "tools", "evaluator": "evaluator"}
)

graph_builder.add_edge("tools", "worker")

graph_builder.add_conditional_edges(
    "evaluator",
    route_after_evaluation,
    {"worker": "worker", "END": END}
)

graph = graph_builder.compile(checkpointer=memory)

In [ ]:

config = {"configurable": {"thread_id": str(uuid.uuid4())}}

state = {
    "messages": [HumanMessage(content="Research the capital of France and explain briefly")],
    "success_criteria": "Answer should be correct and concise",
    "feedback_on_work": None,
    "success_criteria_met": False,
    "user_input_needed": False,
    "plan": None,
    "final_worker_response": None,
}

result = graph.invoke(state, config=config)

print(result["final_worker_response"])

In [ ]:
def run_agent(user_input):
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}

    state = {
        "messages": [HumanMessage(content=user_input)],
        "success_criteria": "Answer should be correct and concise",
        "feedback_on_work": None,
        "success_criteria_met": False,
        "user_input_needed": False,
        "plan": None,
        "final_worker_response": None,
    }

    result = graph.invoke(state, config=config)

    return result["final_worker_response"]

In [ ]:

demo = gr.Interface(
    fn=run_agent,
    inputs=gr.Textbox(lines=3, placeholder="Ask something..."),
    outputs="text",
    title="Agentic Sidekick",
    description="Planner + Worker + Tools + Evaluator loop"
)

demo.launch()